# ZERO STUDIO — Kaggle GPU preflight

This notebook only inspects the runtime. It does **not** clone AniDoc, download model weights, install packages, upload artwork, or run inference.

In Kaggle Notebook settings, select **Accelerator → GPU (T4x2)** when available, and enable Internet only if Kaggle asks for it. Then run the code cell. GPU availability and quotas can vary; do not purchase anything for this test.

In [ ]:
import json, os, platform, shutil, subprocess, sys

report = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'torch_installed': False,
    'torch_version': None,
    'torch_cuda_available': False,
    'gpus': [],
    'disk': {},
    'nvidia_smi': None,
}

try:
    import torch
    report['torch_installed'] = True
    report['torch_version'] = torch.__version__
    report['torch_cuda_available'] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            with torch.cuda.device(i):
                free_b, total_b = torch.cuda.mem_get_info()
            report['gpus'].append({
                'index': i,
                'name': torch.cuda.get_device_name(i),
                'total_vram_gib': round(total_b / 1024**3, 2),
                'free_vram_gib': round(free_b / 1024**3, 2),
            })
except Exception as exc:
    report['torch_error'] = f'{type(exc).__name__}: {exc}'

for path in ['/kaggle/working', '/kaggle/temp', '/tmp']:
    if os.path.exists(path):
        usage = shutil.disk_usage(path)
        report['disk'][path] = {
            'free_gib': round(usage.free / 1024**3, 2),
            'total_gib': round(usage.total / 1024**3, 2),
        }

try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits'],
        capture_output=True, text=True, timeout=15, check=True
    )
    report['nvidia_smi'] = result.stdout.strip()
except Exception as exc:
    report['nvidia_smi_error'] = f'{type(exc).__name__}: {exc}'

gpu_14g_candidate = any(g.get('total_vram_gib', 0) >= 14 for g in report['gpus'])
report['single_gpu_14g_candidate'] = gpu_14g_candidate
report['decision'] = (
    'GPU_CANDIDATE_FOUND — check disk and Python compatibility next; this is not inference readiness.'
    if report['torch_cuda_available'] and gpu_14g_candidate
    else 'BLOCKED — no visible CUDA GPU with at least 14 GiB on one device.'
)

print(json.dumps(report, indent=2))

## What this result tells us

- We need at least one visible GPU with about 14 GiB **on that one GPU**; two 16 GB cards do not automatically combine into one 32 GB memory pool.
- The upstream launcher currently selects GPU 0, so the first 16 GB T4 is the relevant device unless we intentionally change and test the launcher.
- This check does not prove that the model will fit or run. We must still verify Python 3.8 compatibility, actual free VRAM, disk space, and the exact FP16 checkpoint files before downloading anything.
- Copy the JSON output back into the ZERO STUDIO conversation. Do not upload the character reference or model weights to GitHub.